# ⚡ Z-Fooocus
**Multi-Model Studio** — Z-Image Turbo · FLUX.2-klein · Qwen-Image-Edit

Generate · Img2Img · Inpaint with model switching

In [ ]:
# ── Step 1: Install & Download Models ─────────────────────
import os

# ComfyUI
if not os.path.exists('/content/ComfyUI'):
    !git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
    %cd /content/ComfyUI
    !pip install -q -r requirements.txt

# ComfyUI-GGUF custom nodes (for Qwen-Image-Edit GGUF)
if not os.path.exists('/content/ComfyUI/custom_nodes/ComfyUI-GGUF'):
    !git clone https://github.com/city96/ComfyUI-GGUF.git /content/ComfyUI/custom_nodes/ComfyUI-GGUF
    !pip install -q -r /content/ComfyUI/custom_nodes/ComfyUI-GGUF/requirements.txt

# Extra deps
!pip install -q rembg[gpu] onnxruntime-gpu

# Model dirs
UNET = '/content/ComfyUI/models/unet'
CLIP = '/content/ComfyUI/models/clip'
VAE  = '/content/ComfyUI/models/vae'
os.makedirs(UNET, exist_ok=True)
os.makedirs(CLIP, exist_ok=True)
os.makedirs(VAE,  exist_ok=True)

# ── Z-Image Turbo FP8 ─────────────────────────────────────
if not os.path.exists(f'{UNET}/z-image-turbo-fp8-e4m3fn.safetensors'):
    print('⏳ Downloading Z-Image Turbo FP8...')
    !wget -q --show-progress -O {UNET}/z-image-turbo-fp8-e4m3fn.safetensors \
        https://huggingface.co/Tongyi-MAI/Z-Image-Turbo/resolve/main/z-image-turbo-fp8-e4m3fn.safetensors

# Qwen 3.4B CLIP (for Z-Image Turbo + FLUX)
if not os.path.exists(f'{CLIP}/qwen_3_4b.safetensors'):
    print('⏳ Downloading Qwen 3.4B CLIP...')
    !wget -q --show-progress -O {CLIP}/qwen_3_4b.safetensors \
        https://huggingface.co/Comfy-Org/Qwen_3_4B_Lumina2_CLIP/resolve/main/qwen_3_4b.safetensors

# VAE for Z-Image Turbo + FLUX
if not os.path.exists(f'{VAE}/ae.safetensors'):
    print('⏳ Downloading VAE (ae.safetensors)...')
    !wget -q --show-progress -O {VAE}/ae.safetensors \
        https://huggingface.co/Comfy-Org/flux1-dev/resolve/main/ae.safetensors

# ── FLUX.2-klein FP8 (~10GB) ──────────────────────────────
if not os.path.exists(f'{UNET}/flux-2-klein-9b-kv-fp8.safetensors'):
    print('⏳ Downloading FLUX.2-klein 9B FP8 (~10GB)...')
    !wget -q --show-progress -O {UNET}/flux-2-klein-9b-kv-fp8.safetensors \
        https://huggingface.co/black-forest-labs/FLUX.2-klein-9b-kv-fp8/resolve/main/flux-2-klein-9b-kv-fp8.safetensors

# ── Qwen-Image-Edit-2511 GGUF Q4_K_M ─────────────────────
# From: https://huggingface.co/unsloth/Qwen-Image-Edit-2511-GGUF
if not os.path.exists(f'{UNET}/qwen-image-edit-2511-Q4_K_M.gguf'):
    print('⏳ Downloading Qwen-Image-Edit-2511 Q4_K_M GGUF...')
    !wget -q --show-progress -O {UNET}/qwen-image-edit-2511-Q4_K_M.gguf \
        https://huggingface.co/unsloth/Qwen-Image-Edit-2511-GGUF/resolve/main/qwen-image-edit-2511-Q4_K_M.gguf

# Qwen2.5-VL CLIP GGUF (text encoder for Qwen-Image-Edit)
if not os.path.exists(f'{CLIP}/Qwen2.5-VL-7B-Instruct-UD-Q4_K_XL.gguf'):
    print('⏳ Downloading Qwen2.5-VL CLIP GGUF...')
    !wget -q --show-progress -O {CLIP}/Qwen2.5-VL-7B-Instruct-UD-Q4_K_XL.gguf \
        https://huggingface.co/unsloth/Qwen2.5-VL-7B-Instruct-GGUF/resolve/main/Qwen2.5-VL-7B-Instruct-UD-Q4_K_XL.gguf

# Qwen-Image VAE
if not os.path.exists(f'{VAE}/qwen_image_vae.safetensors'):
    print('⏳ Downloading Qwen-Image VAE...')
    !wget -q --show-progress -O {VAE}/qwen_image_vae.safetensors \
        https://huggingface.co/Qwen/Qwen-Image-Edit-2511/resolve/main/vae/diffusion_pytorch_model.safetensors

print('\n✅ All models ready!')
print('   Z-Image Turbo FP8     ✅')
print('   FLUX.2-klein FP8      ✅')
print('   Qwen-Image-Edit GGUF  ✅')

In [ ]:
# ── Step 2: Launch Z-Fooocus ──────────────────────────────
import os
os.chdir('/content')
REPO = 'https://github.com/MuntasirMalek/Z-Fooocus.git'
if os.path.exists('/content/Z-Fooocus'):
    %cd /content/Z-Fooocus
    !git pull --ff-only || true
else:
    !git clone {REPO}
    %cd /content/Z-Fooocus
!python app.py